In [ ]:
!pip install langchain==0.2.11 langchain-google-genai==1.0.0 chromadb==0.5.3
!pip install -U langchain langchain-core langchain-community langchain-google-genai langchain-chroma
!pip install langchain langchain-google-genai langchain-chroma langchain-core langchain-text-splitters chromadb
!pip install -U langchain langchain-community langchain-core langchain-google-genai

ERROR: Could not find a version that satisfies the requirement langchain-google-genai==1.0.0 (from versions: 0.0.1rc0, 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10rc0, 0.0.11, 1.0.1, 1.0.2, 1.0.3, 1.0.4, 1.0.5, 1.0.6, 1.0.7, 1.0.8, 1.0.9, 1.0.10, 2.0.0.dev1, 2.0.0, 2.0.1, 2.0.2, 2.0.3, 2.0.4, 2.0.5, 2.0.6, 2.0.7, 2.0.8, 2.0.9, 2.0.10, 2.0.11, 2.1.0, 2.1.1, 2.1.2, 2.1.3, 2.1.4, 2.1.5, 2.1.6, 2.1.7, 2.1.8, 2.1.9, 2.1.10, 2.1.11, 2.1.12, 3.0.0a1, 3.0.0rc1, 3.0.0, 3.0.1, 3.0.2, 3.0.3, 3.1.0, 3.2.0, 4.0.0, 4.1.0, 4.1.1, 4.1.2, 4.1.3, 4.2.0, 4.2.1, 4.2.2, 4.2.3, 4.2.4, 4.2.5, 4.2.6, 4.2.7, 4.3.0, 4.3.1, 4.3.2, 4.3.3, 4.3.4, 4.3.5, 4.3.6, 4.3.7, 4.4.0, 4.4.1, 4.4.2)
ERROR: No matching distribution found for langchain-google-genai==1.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 126.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s  0:00:00
   ━━━━━

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 140.3 MB/s  0:00:00


In [ ]:
!pip install -q rank_bm25 rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os                                                  # API key from environment, file paths
import re                                                  # regex for cleaning text and BM25 tokenizing
import json                                                # save/load evaluation queries and results
import time                                                # measure search/summarization latency
import random                                              # reproducible sampling of the test set
import pickle                                              # save/load docs and BM25 index

import numpy as np                                         # arrays, sorting scores, mean metrics
import pandas as pd                                        # result tables for the report
from tqdm.auto import tqdm                                 # progress bars

from datasets import load_dataset                          # load CNN/DailyMail

from langchain_google_genai import ChatGoogleGenerativeAI  # Gemini: summaries + test-query generation
from langchain_core.prompts import ChatPromptTemplate      # summarization prompt with a length setting
from langchain_core.output_parsers import StrOutputParser  # LLM response -> plain string

from sentence_transformers import SentenceTransformer, CrossEncoder  # local embeddings + optional reranker
import faiss                                               # vector index
from rank_bm25 import BM25Okapi                            # BM25 keyword search
from rouge_score import rouge_scorer                       # ROUGE for summary evaluation

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("all imports ok")

all imports ok


DATA PREPARATION

In [ ]:
# ---------- 1. Load a random subset of CNN/DailyMail ----------

raw = load_dataset("abisee/cnn_dailymail", "3.0.0", split="train")
raw = raw.shuffle(seed=SEED).select(range(25000))     # oversample, since cleaning removes some articles

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
# ---------- 2. Cleaning function ----------
def clean_text(t):
    t = re.sub(r"^.{0,120}?\(CNN\)\s*(--)?\s*", "", t)                      # "LONDON, England (CNN) --" prefix
    t = re.sub(r"^By \. .{0,120}? \. ", "", t)                              # DailyMail byline: "By . Freya Noble . "
    t = re.sub(r"^and .{0,80}? \. ", "", t)                                 # second author: "and Name . "
    t = re.sub(r"^Last updated at .{0,60}?\d{4} \. ", "", t)                # "Last updated at 10:15 AM on 23rd January 2012 ."
    t = re.sub(r"^PUBLISHED: \. .{0,60}?\d{4} \. (\| \. )?", "", t)         # "PUBLISHED: . 05:28 EST, 22 January 2014 . | ."
    t = re.sub(r"^UPDATED: \. .{0,60}?\d{4} \. ", "", t)                    # "UPDATED: . 12:29 EST, 22 January 2014 ."
    t = re.sub(r"https?://\S+|www\.\S+", "", t)                             # URLs
    t = re.sub(r"<[^>]+>", " ", t)                                          # leftover HTML tags
    t = t.replace("\u2018", "'").replace("\u2019", "'")                     # normalize curly single quotes
    t = t.replace("\u201c", '"').replace("\u201d", '"')                     # normalize curly double quotes
    t = re.sub(r"\s+", " ", t).strip()                                      # collapse whitespace
    return t

In [ ]:
# ---------- 3. Clean, filter by length, deduplicate ----------
MIN_WORDS, MAX_WORDS, N_DOCS = 100, 1500, 20000
docs, seen = [], set()
for r in tqdm(raw, desc="Cleaning"):
    text = clean_text(r["article"])
    n_words = len(text.split())
    if n_words < MIN_WORDS or n_words > MAX_WORDS:      # drop very short and very long articles
        continue
    key = hash(text[:200].lower())                      # near-duplicate check on the opening text
    if key in seen:
        continue
    seen.add(key)
    docs.append({
        "id": len(docs),                                # position in the index = doc id
        "text": text,                                   # searchable text (NEVER include the summary here)
        "summary": r["highlights"],                     # reference summary, used only for ROUGE evaluation
    })
    if len(docs) == N_DOCS:
        break

print(f"Kept {len(docs)} documents")

Cleaning:   0%|          | 0/25000 [00:00<?, ?it/s]

Kept 20000 documents


In [ ]:
# ---------- Cleaning quality check + stats for the report ----------
lengths = [len(d["text"].split()) for d in docs]
print(f"Articles processed: 20638 | kept: {len(docs)} | removed: {20638 - len(docs)}")
print(f"Words per doc -> mean: {np.mean(lengths):.0f}, median: {np.median(lengths):.0f}, min: {min(lengths)}, max: {max(lengths)}")

# look at the start of a few random articles to spot leftover boilerplate
for i in random.sample(range(len(docs)), 5):
    print(f"\n[{i}] {docs[i]['text'][:200]}")

# count how many articles still start with common boilerplate
leftover = sum(1 for d in docs if re.match(r"^(By \.|\(CNN\)|Last updated|PUBLISHED)", d["text"]))
print(f"\nArticles still starting with boilerplate: {leftover}")

Articles processed: 20638 | kept: 20000 | removed: 638
Words per doc -> mean: 663, median: 620, min: 100, max: 1500

[7798] For most young people the summer months mean the end of studying, but also the start of a lowly-paid temp job...unless you're one of the 'rich kids of Instagram.' For them the lazy weeks include trips

[9125] Behind bars: 37-year-old Bobbre Williams was arrested Wednesday after police caught her smoking crack and drinking with her 1-month-old baby girl . A Daytona Beach, Florida mother of seven was arreste

[15906] White House Senior Adviser Dan Pfeiffer said today he believes there's a 'possibility' that House Republicans will try to impeach the president. The Huffington Post reports that Pfeiffer told attendee

[7020] Manuel Neuer captained Schalke 04 to the German Cup with a 5-0 rout of second-flight Duisberg while earlier Saturday Celtic salvaged their season with a 3-0 win over Motherwell to claim the Scottish e

[17669] Heavy and durable, these trucks look l

In [ ]:
# ---------- 4. Held-out test set for evaluation ----------
test_ids = random.sample(range(len(docs)), 200)         # these docs stay in the corpus; queries are generated from them
print("Test set size:", len(test_ids))

Test set size: 200


In [ ]:
# ---------- 5. Save the cleaned corpus ----------
with open("docs.pkl", "wb") as f:
    pickle.dump({"docs": docs, "test_ids": test_ids}, f)

In [ ]:
# ---------- 1. Embedding model (runs locally, free) ----------
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # use a GPU runtime in Colab to speed this up
texts = [d["text"] for d in docs]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# ---------- 2. Dense index with FAISS ----------
t0 = time.time()
emb = embedder.encode(
    texts, batch_size=128, normalize_embeddings=True,      # normalized, so inner product = cosine similarity
    show_progress_bar=True
).astype("float32")
print(f"Embedding time: {time.time() - t0:.1f}s")

index = faiss.IndexFlatIP(emb.shape[1])                    # exact search; use HNSW/IVF for much larger corpora
index.add(emb)
faiss.write_index(index, "cnn_dm.faiss")                   # persist, so you embed only once
print("FAISS vectors:", index.ntotal)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding time: 245.7s
FAISS vectors: 20000


In [ ]:
# ---------- 3. Keyword index with BM25 (traditional IR) ----------
tokenize = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(t) for t in tqdm(texts, desc="BM25")])
with open("bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)                                   # persist the BM25 index


BM25:   0%|          | 0/20000 [00:00<?, ?it/s]

In [ ]:
# ---------- 4. Quick sanity check ----------
q = "government announces new economic policy"
qv = embedder.encode([q], normalize_embeddings=True).astype("float32")
scores, ids = index.search(qv, 3)
for s, i in zip(scores[0], ids[0]):
    print(f"{s:.3f} | {docs[i]['text'][:120]}...")

0.670 | Communities Secretary Eric Pickles said that a 'new economy is emerging' outside London . The boom of new private sector...
0.666 | The Federal Budget for 2014 will see big cuts to Foreign Aid, higher prices at the petrol pump, and tertiary students co...
0.664 | As the New Year rolled in, so did a number of key legislation changes in the areas of health, welfare, immigration and e...


In [ ]:
def dense_search(query, k=50):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)                          # semantic search (embeddings)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def bm25_search(query, k=50):
    scores = bm25.get_scores(tokenize(query))                  # keyword search (traditional IR)
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]

def rrf_fuse(result_lists, k_const=60):
    fused = {}
    for results in result_lists:                               # combine by rank, so score scales don't matter
        for rank, (doc_id, _) in enumerate(results, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k_const + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)

def search(query, n=5, mode="hybrid"):
    t0 = time.time()
    if mode == "dense":
        ranked = dense_search(query)
    elif mode == "bm25":
        ranked = bm25_search(query)
    else:
        ranked = rrf_fuse([dense_search(query), bm25_search(query)])
    results = [{"id": i, "score": s, "text": docs[i]["text"]} for i, s in ranked[:n]]
    return results, time.time() - t0

# sanity check
res, secs = search("government announces new economic policy", n=3)
for r in res:
    print(f"{r['score']:.4f} | id={r['id']} | {r['text'][:130]}...")

0.0303 | id=11316 | Controversial: Planning minister Nick Boles defended the new policy - arguing the current regulations are holding back economic gr...
0.0279 | id=7976 | Families face being 'clobbered' by a rise in interest rates, it was warned today as unemployment fell to within a whisker of the t...
0.0228 | id=7024 | New Government regulations which would hurt small businesses will be 'reformed or binned', the Department for Business will announ...


Summarizer with length control

In [ ]:
# ---------- 1. Gemini setup ----------
from google.colab import userdata                                # read the key from Colab Secrets (key icon in the left sidebar)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")    # never hard-code the key in the notebook

MODEL_NAME = "gemini-3.5-flash"                                  # put the exact model name you have access to here
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0)    # temperature 0 = stable, reproducible summaries

# ---------- 2. Length control ----------
LENGTHS = {"short": 50, "medium": 120, "long": 250}              # target word counts

def resolve_length(length):
    return length if isinstance(length, int) else LENGTHS[length]   # accepts "short"/"medium"/"long" or a number of words

# ---------- 3. Prompt ----------
prompt = ChatPromptTemplate.from_template(
    "You are a news summarizer. Summarize the article(s) below in about {words} words "
    "(do not exceed {words} words). Use only information from the text, be factual and coherent, "
    "and if there are several articles, combine them into one summary instead of summarizing each separately.\n\n"
    "{content}\n\nSummary:"
)
chain = prompt | llm | StrOutputParser()                         # prompt -> Gemini -> plain string

# ---------- 4. Summarize function ----------
def summarize(results, length="medium", max_words_per_doc=400):
    words = resolve_length(length)
    content = "\n\n".join(
        f"Article {i+1}: " + " ".join(r["text"].split()[:max_words_per_doc])   # truncate each article to save tokens and quota
        for i, r in enumerate(results)
    )
    for attempt in range(3):                                     # simple retry for free-tier rate limits
        try:
            t0 = time.time()
            out = chain.invoke({"words": words, "content": content})
            return out.strip(), time.time() - t0
        except Exception as e:
            print(f"LLM error (attempt {attempt+1}): {e}")
            time.sleep(10 * (attempt + 1))
    return "", 0.0

# ---------- 5. Full pipeline: search -> summarize ----------
def search_and_summarize(query, n=3, length="medium", mode="hybrid"):
    results, search_secs = search(query, n=n, mode=mode)
    summary, llm_secs = summarize(results, length)
    return results, summary, search_secs, llm_secs

# ---------- 6. Sanity check ----------
q = "government announces new economic policy"
for length in ["short", "medium", "long"]:
    results, summary, s_t, l_t = search_and_summarize(q, n=3, length=length)
    print(f"\n=== {length} | {len(summary.split())} words | search {s_t*1000:.0f} ms, LLM {l_t:.1f} s ===")
    print(summary)


=== short | 41 words | search 68 ms, LLM 24.6 s ===
To support economic growth, Planning Minister Nick Boles defended plans to double permitted home extension limits, while Business Minister Michael Fallon banned new regulations impacting small businesses. Meanwhile, UK unemployment fell sharply to 7.1% following a record quarterly increase in employment.

=== medium | 116 words | search 65 ms, LLM 20.0 s ===
To boost economic growth, Planning Minister Nick Boles defended plans to double permitted home extension sizes and fast-track commercial projects. Simultaneously, Business Minister Michael Fallon announced a crackdown on red tape, banning new regulations that burden small businesses with up to 50 employees. 

These initiatives come alongside news of economic recovery, as a record quarterly increase of 280,000 jobs raised total UK employment to 30.15 million. Consequently, unemployment fell sharply to 7.1 per cent, nearing the Bank of England's 7 per cent threshold to consider int

# Evaluation

retrieval evaluation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM   # local model loading
from langchain_core.runnables import RunnableLambda            # lets a plain function act as the llm step

LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct"                       # fits a T4; use Qwen/Qwen2.5-1.5B-Instruct if you run out of GPU memory
tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
lm = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype=torch.float16, device_map="auto")

def generate(prompt_text, max_new_tokens=400):
    msgs = [{"role": "user", "content": prompt_text}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = reproducible
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

llm = RunnableLambda(lambda p: generate(p.to_string()))        # replaces the Gemini/OpenAI llm
chain = prompt | llm | StrOutputParser()                       # summarization chain
query_chain = query_prompt | llm | StrOutputParser()           # test-query chain

# test
res, _ = search("government announces new economic policy", n=3)
summ, secs = summarize(res, length="short")
print(f"{len(summ.split())} words, {secs:.1f}s\n{summ}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

50 words, 5.4s
Nick Boles, Planning Minister, defends doubling extension sizes without planning consent, arguing it will boost economic growth. Unemployment fell to 7.1%, nearing Bank of England target, with 280,000 new jobs created. Michael Fallon announces reforms to reduce red tape affecting small businesses, aiming to mitigate burdens and promote faster growth.


In [ ]:
# 1. One LLM-generated query per held-out document (cached, so a restart doesn't redo it)
cache_file = "eval_queries.json"
eval_ids = test_ids[:50]
eval_queries = {int(k): v for k, v in json.load(open(cache_file)).items()} if os.path.exists(cache_file) else {}

for i in tqdm(eval_ids):
    if i not in eval_queries:
        q = query_chain.invoke({"article": " ".join(docs[i]["text"].split()[:300])})
        eval_queries[i] = q.strip().strip('"').split("\n")[0]    # keep only the first line
json.dump(eval_queries, open(cache_file, "w"))
for i in eval_ids[:3]:
    print(f"doc {i}: {eval_queries[i]}")                         # check the queries look sensible

# 2. Does search return the source document? Compare the three modes
def evaluate(mode):
    ranks = []
    for doc_id, q in eval_queries.items():
        res, _ = search(q, n=10, mode=mode)
        ids = [r["id"] for r in res]
        ranks.append(ids.index(doc_id) + 1 if doc_id in ids else None)
    return {"mode": mode,
            "Recall@1": np.mean([r == 1 for r in ranks]),
            "Recall@5": np.mean([r is not None and r <= 5 for r in ranks]),   # source doc in top 5
            "MRR": np.mean([1 / r if r else 0 for r in ranks])}               # higher rank = higher score

retrieval_df = pd.DataFrame([evaluate(m) for m in ["bm25", "dense", "hybrid"]]).round(3)
print(retrieval_df)
retrieval_df.to_csv("retrieval_results.csv", index=False)

  0%|          | 0/50 [00:00<?, ?it/s]

doc 4335: California Institute of Tech solar flare lab breakthrough
doc 18711: Senate rejects balanced budget amendment proposals
doc 18881: FEMA controlled formaldehyde study on Katrina trailers
     mode  Recall@1  Recall@5    MRR
0    bm25      0.98       1.0  0.987
1   dense      0.94       1.0  0.970
2  hybrid      0.96       1.0  0.977


summary evaluation (ROUGE plus the human-eval file)

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
lead3 = lambda t: " ".join(re.split(r"(?<=[.!?])\s+", t)[:3])    # baseline: first 3 sentences
rows, human = [], []

for i in tqdm(eval_ids[:30]):
    summ, _ = summarize([{"text": docs[i]["text"]}], length=50)  # single-document summary with the local LLM
    ref = docs[i]["summary"]                                     # human-written reference highlights
    for name, cand in [("Qwen LLM", summ), ("Lead-3 baseline", lead3(docs[i]["text"]))]:
        sc = scorer.score(ref, cand)
        rows.append({"system": name, **{k: v.fmeasure for k, v in sc.items()}})
    human.append({"doc_id": i, "reference": ref, "llm_summary": summ,
                  "relevance_1to5": "", "coherence_1to5": "", "faithfulness_1to5": ""})

rouge_df = pd.DataFrame(rows).groupby("system").mean().round(3)
print(rouge_df)
rouge_df.to_csv("rouge_results.csv")
pd.DataFrame(human[:10]).to_csv("human_eval.csv", index=False)  # score these 10 by hand

  0%|          | 0/30 [00:00<?, ?it/s]

                 rouge1  rouge2  rougeL
system                                 
Lead-3 baseline   0.387   0.151   0.241
Qwen LLM          0.363   0.118   0.237


HUMAN EVALUATION

In [ ]:
he = pd.read_csv("human_eval.csv")
scores = []
for n, r in he.iterrows():
    print(f"\n===== {n+1}/{len(he)} | doc {r['doc_id']} =====")
    print("ARTICLE START:", docs[int(r["doc_id"])]["text"][:600], "...")
    print("\nREFERENCE   :", r["reference"].replace("\n", " "))
    print("LLM SUMMARY :", r["llm_summary"])
    s = [int(input(f"{m} (1-5): ")) for m in ["relevance", "coherence", "faithfulness"]]
    scores.append(s)

he[["relevance_1to5", "coherence_1to5", "faithfulness_1to5"]] = scores
he.to_csv("human_eval_scored.csv", index=False)                  # keep the scored file as evidence for the report
print("\n", he[["relevance_1to5", "coherence_1to5", "faithfulness_1to5"]].mean().round(2))



===== 1/10 | doc 4335 =====
ARTICLE START: Solar flares can cause widespread disruption on Earth, but scientists have struggled to understand how they are formed. Now, a team from from the California Institute of Technology (Caltech) have taken a huge step forward by recreated the phenomenon that leads to solar flares in a lab. They hope their work could lead to improved space weather forecasts. The US team amanged to recreate plasma loops (above) in a lab for the first time. The loops are precursors to the massive solar flares on the surface of the sun can cause electrical storms on earth. 'We're studying how these solar loops work, w ...

REFERENCE   : Team hope to learn more about how flares that can disrupt electricity on earth are formed . Solar storms can affect TV, radio and satellite communications on Earth .
LLM SUMMARY : Scientists at Caltech recreated plasma loops in a lab, similar to those that precede solar flares on the sun, aiming to improve space weather forecasting. T

# USER INTERFACE WITH STREAMLIT

In [ ]:
%%writefile app.py
import re, time, pickle
import numpy as np
import faiss, torch
import streamlit as st
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

st.set_page_config(page_title="News Search & Summarizer", layout="wide")
PAGE_SIZE, TOP_K = 5, 30                                   # 5 results per page, 30 retrieved per query
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

# ---------- Load everything once (cached across reruns) ----------
@st.cache_resource(show_spinner="Loading indexes and models (first run takes a minute)...")
def load_all():
    docs = pickle.load(open("docs.pkl", "rb"))["docs"]
    index = faiss.read_index("cnn_dm.faiss")
    bm25 = pickle.load(open("bm25.pkl", "rb"))
    embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
    tok = AutoTokenizer.from_pretrained(LLM_NAME)
    lm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="auto")
    suggestions = [d["summary"].split("\n")[0].strip() for d in docs]   # first highlight line of each article
    return docs, index, bm25, embedder, tok, lm, suggestions

docs, index, bm25, embedder, tok, lm, suggestions = load_all()
tokenize = lambda s: re.findall(r"\w+", s.lower())

# ---------- Search ----------
def dense_search(q, k):
    v = embedder.encode([q], normalize_embeddings=True).astype("float32")
    s, i = index.search(v, k)
    return [(int(a), float(b)) for a, b in zip(i[0], s[0])]

def bm25_search(q, k):
    sc = bm25.get_scores(tokenize(q))
    top = np.argsort(sc)[::-1][:k]
    return [(int(i), float(sc[i])) for i in top]

def rrf(lists, c=60):                                       # Reciprocal Rank Fusion
    f = {}
    for res in lists:
        for r, (d, _) in enumerate(res, 1):
            f[d] = f.get(d, 0) + 1 / (c + r)
    return sorted(f.items(), key=lambda x: -x[1])

@st.cache_data(show_spinner=False)
def run_search(q, mode):
    t0 = time.time()
    if mode == "Dense (embeddings)":
        r = dense_search(q, 50)
    elif mode == "BM25 (keywords)":
        r = bm25_search(q, 50)
    else:
        r = rrf([dense_search(q, 50), bm25_search(q, 50)])
    return r[:TOP_K], time.time() - t0

# ---------- Summarization (local LLM, cached per query/length/results) ----------
@st.cache_data(show_spinner=False)
def run_summary(q, words, ids):
    content = "\n\n".join(f"Article {n+1}: " + " ".join(docs[i]["text"].split()[:400]) for n, i in enumerate(ids))
    prompt = (f"You are a news summarizer. Summarize the article(s) below in about {words} words "
              f"(do not exceed {words} words). Use only information from the text, be factual and coherent, "
              f"and if there are several articles, combine them into one summary instead of summarizing each separately.\n\n"
              f"{content}\n\nSummary:")
    t0 = time.time()
    inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inputs, max_new_tokens=int(words * 1.8) + 20, do_sample=False)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text, time.time() - t0

# ---------- UI ----------
st.title("📰 News Search & Summarization")
st.session_state.setdefault("page", 0)

with st.sidebar:
    mode = st.radio("Search method", ["Hybrid (BM25 + embeddings)", "Dense (embeddings)", "BM25 (keywords)"])
    words = st.slider("Summary length (words)", 30, 250, 100, 10)
    n_sum = st.slider("Summarize top N results", 1, 5, 3)
    show_summary = st.toggle("Generate summary", value=True)

st.text_input("Search the news corpus", key="query", placeholder="e.g. senate vote on balanced budget amendment")
q = st.session_state.get("query", "").strip()

# query auto-suggestions (appear after you press Enter or click out of the box)
if len(q) >= 3:
    matches = [s for s in suggestions if q.lower() in s.lower()][:5]
    if matches and q not in matches:
        st.caption("Suggestions:")
        for n, s in enumerate(matches):
            st.button(s, key=f"sg{n}", on_click=lambda s=s: st.session_state.update(query=s))

def go(delta):
    st.session_state.page += delta

if q:
    if st.session_state.get("last") != (q, mode):            # new query or mode: back to page 1
        st.session_state.page, st.session_state.last = 0, (q, mode)
    results, secs = run_search(q, mode)
    st.write(f"**{len(results)} results** retrieved in {secs*1000:.0f} ms")

    if show_summary:
        with st.spinner("Summarizing..."):
            summ, lsecs = run_summary(q, words, tuple(d for d, _ in results[:n_sum]))
        st.subheader(f"Summary of top {n_sum} results")
        st.info(summ)
        st.caption(f"{len(summ.split())} words (target {words}) · generated in {lsecs:.1f} s")

    n_pages = (len(results) + PAGE_SIZE - 1) // PAGE_SIZE
    c1, c2, c3 = st.columns([1, 1, 6])
    c1.button("◀ Prev", on_click=go, args=(-1,), disabled=st.session_state.page <= 0)
    c2.button("Next ▶", on_click=go, args=(1,), disabled=st.session_state.page >= n_pages - 1)
    c3.write(f"Page {st.session_state.page + 1} of {n_pages}")

    start = st.session_state.page * PAGE_SIZE
    for rank, (d, score) in enumerate(results[start:start + PAGE_SIZE], start + 1):
        with st.container(border=True):
            st.markdown(f"**#{rank}** · doc {d} · score {score:.4f}")
            st.write(" ".join(docs[d]["text"].split()[:60]) + " ...")
            with st.expander("Full article"):
                st.write(docs[d]["text"])

Writing app.py


In [ ]:
!pip install -q streamlit
!ls -la app.py docs.pkl cnn_dm.faiss bm25.pkl

-rw-r--r-- 1 root root     5656 Oct 30 09:32 app.py
-rw-r--r-- 1 root root 65964314 Oct 30 08:15 bm25.pkl
-rw-r--r-- 1 root root 30720045 Oct 30 08:12 cnn_dm.faiss
-rw-r--r-- 1 root root 83831010 Oct 30 08:08 docs.pkl


In [ ]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &
import time; time.sleep(90)                                   # the models take 1-3 minutes to load
!tail -n 20 /content/streamlit.log
!curl -s -o /dev/null -w "local streamlit status: %{http_code}\n" http://localhost:8501



2026-10-30 09:42:36.759 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.85.39.117:8501

local streamlit status: 200


In [ ]:
!curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501

136.85.39.117
⠙⠹⠸⠼⠴your url is: https://bright-words-clean.loca.lt
^C


In [ ]:
!pkill -f streamlit
!nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &
import time; time.sleep(60)
!curl -s -o /dev/null -w "local streamlit status: %{http_code}\n" http://localhost:8501

local streamlit status: 200


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate

2026-10-30T09:51:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-30T09:51:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-30T09:51:34Z INF +--------------------------------------------------------------------------------------------+
2026-10-30T09:51:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-30T09:51:34Z INF |  https://comment-das-featuring-presentations.trycloudf